In [1]:
import os, json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv()

True

In [2]:
# I. need to load the data
with open("bajaj_db.json") as f:
    db = json.load(f)



In [6]:
@tool
def get_loan_status(loan_id: str) -> dict:
    """Fetches current status of a Bajaj Finance loan from the database.
    Returns EMI amount, remaining months, outstanding balance, next due date.
    Use this when customer asks about their loan details, EMI, or balance.
    Args:
        loan_id: The loan account number (e.g., 'BFL2024001')
    """
    if loan_id not in db["loans"]:
        return {"error": f"Loan {loan_id} not found"}
    loan = db["loans"][loan_id]
    return {
        "customer_name": loan["customer_name"],
        "emi": loan["emi"],
        "remaining_months": loan["remaining_months"],
        "outstanding": loan["outstanding"],
        "next_due_date": loan["next_due_date"],
        "prepayment_charge_pct": loan["prepayment_charge_pct"]
    }


In [8]:
get_loan_status.invoke('BFL9988')

{'customer_name': 'Priya Mehta',
 'emi': 24500,
 'remaining_months': 204,
 'outstanding': 2180000,
 'next_due_date': '2026-05-10',
 'prepayment_charge_pct': 0.0}

In [9]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# I can connect my llm with tools
tools = [get_loan_status]

llm_with_tools =llm.bind_tools(tools)

In [31]:
response =llm_with_tools.invoke('what is my emi for loan id BFL9988')

In [ ]:
response.content

''

In [ ]:
response.tool_calls

[{'name': 'get_loan_status',
  'args': {'loan_id': 'BFL9988'},
  'id': 'call_dsrsPis3CVpRzfPG1jKvD7Jc',
  'type': 'tool_call'}]

In [32]:
tool_map ={"get_loan_status":get_loan_status}

tool_result = []


for tc in response.tool_calls:
    tool_name = tc["name"]
    print("tool name: ",tool_name)
    tool_args = tc["args"]
    print("tool_args ",tool_args)
    tool_id = tc["id"]
    result = tool_map[tool_name].invoke(tool_args)
    tool_result.append((tool_id,tool_name,result))


tool name:  get_loan_status
tool_args  {'loan_id': 'BFL9988'}


In [33]:
tool_result

[('call_vaVFkXepsnN4sevf9B0Wmjrx',
  'get_loan_status',
  {'customer_name': 'Priya Mehta',
   'emi': 24500,
   'remaining_months': 204,
   'outstanding': 2180000,
   'next_due_date': '2026-05-10',
   'prepayment_charge_pct': 0.0})]

In [39]:
system = SystemMessage(content="You are a Bajaj Finance support agent. Use tools for real data.")
user_msg = HumanMessage(content="what is my emi for loan id BFL2024001")

In [40]:
message = [system,user_msg,response]

for tool_id,tool_name, result in tool_result:
    tool_msg = ToolMessage(
        content = str(result) ,# must be in string
        tool_call_id = tool_id

    )

    message.append(tool_msg)

In [41]:
message

[SystemMessage(content='You are a Bajaj Finance support agent. Use tools for real data.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is my emi for loan id BFL2024001', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 114, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b541469465', 'id': 'chatcmpl-DfchTpHtAO0FE9t2Jv1J0vHIMaWgx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e297d-f896-71c2-b52f-254420cbc5c3-0', tool_calls=[{'name': 'get_loan_status', 'args': {'loan_id': 'BFL9988'}, 'id': 'call_vaVFkXepsnN4sevf9B0

In [42]:
final_response = llm_with_tools.invoke(message)

In [44]:
final_response.content

'Your EMI for loan ID BFL2024001 is ₹24,500. If you have any more questions or need further assistance, feel free to ask!'